# 02 Low-read filter sensitivity analysis

This notebook evaluates how the W/O genotype landscape changes when low-depth sequencing samples are excluded before pooling counts by library and condition.

The baseline dissertation-associated analysis includes all samples present in `wo_counts.csv`. This notebook applies sample-level total UMI count cutoffs, recalculates pooled frequencies and enrichment values, and compares the filtered results against the dissertation-associated landscape table.

The goal is to determine whether the major enrichment patterns are robust to removal of low-depth samples.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Checkpoint output:", CHECKPOINT_DIR)

Project root: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape
Processed data: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/processed
Checkpoint output: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/outputs/checkpoints


In [2]:
wo_counts = pd.read_csv(PROCESSED_DIR / "wo_counts.csv")
baseline_landscape = pd.read_csv(PROCESSED_DIR / "WO_landscape_per_library.csv")

print("wo_counts:", wo_counts.shape)
print("baseline_landscape:", baseline_landscape.shape)

wo_counts: (5451, 6)
baseline_landscape: (768, 11)


In [3]:
sample_coverage = (
    wo_counts
    .groupby(["sample", "library_id", "condition", "replicate"], as_index=False)
    .agg(
        total_umi_count=("umi_count", "sum"),
        n_detected_genotypes=("wo", "nunique")
    )
    .sort_values("total_umi_count")
)

sample_coverage

,sample,library_id,condition,replicate,total_umi_count,n_detected_genotypes
11,Library-2-Pos-2,2,Pos,2,440,155
20,Library-3-Pre-2,3,Pre,2,466,191
13,Library-2-Pre-2,2,Pre,2,2240,248
1,Library-1-Neg-2,1,Neg,2,13848,250
17,Library-3-Pos-1,3,Pos,1,14623,255
7,Library-1-Pre-2,1,Pre,2,89573,256
15,Library-3-Neg-2,3,Neg,2,242077,256
4,Library-1-Pos-2,1,Pos,2,338370,256
6,Library-1-Pre-1,1,Pre,1,3048475,256
16,Library-3-Neg-3,3,Neg,3,3426082,256


In [4]:
def recalculate_landscape_from_counts(
    counts_df: pd.DataFrame,
    epsilon: float = 1e-6
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Recalculate pooled W/O genotype counts and W/O landscape metrics
    from replicate-level genotype counts.
    
    Parameters
    ----------
    counts_df:
        DataFrame with columns:
        wo, umi_count, library_id, condition, replicate, sample
    
    epsilon:
        Pseudocount used for log2 fold-change calculations.
    
    Returns
    -------
    pooled:
        Pooled counts/frequencies by library, condition, and genotype.
    
    landscape:
        Landscape table with frequencies, log2FC values, raw frequency shifts,
        mutational load, and ddelta.
    """
    
    pooled = (
        counts_df
        .groupby(["library_id", "condition", "wo"], as_index=False)
        .agg(umi_count=("umi_count", "sum"))
    )

    pooled["total_cond"] = (
        pooled
        .groupby(["library_id", "condition"])["umi_count"]
        .transform("sum")
    )

    pooled["freq"] = pooled["umi_count"] / pooled["total_cond"]

    pooled = pooled.sort_values(
        ["library_id", "condition", "wo"]
    ).reset_index(drop=True)

    landscape_freq = (
        pooled
        .pivot_table(
            index=["library_id", "wo"],
            columns="condition",
            values="freq",
            fill_value=0
        )
        .reset_index()
    )

    landscape_counts = (
        pooled
        .pivot_table(
            index=["library_id", "wo"],
            columns="condition",
            values="umi_count",
            fill_value=0
        )
        .reset_index()
        .rename(columns={
            "Pre": "count_pre",
            "Pos": "count_pos",
            "Neg": "count_neg",
        })
    )

    landscape = landscape_freq.merge(
        landscape_counts,
        on=["library_id", "wo"],
        how="left"
    )

    for condition in ["Neg", "Pos", "Pre"]:
        if condition not in landscape.columns:
            landscape[condition] = 0.0

    for count_col in ["count_pre", "count_pos", "count_neg"]:
        if count_col not in landscape.columns:
            landscape[count_col] = 0

    landscape["log2fc_pos_pre"] = np.log2(
        (landscape["Pos"] + epsilon) /
        (landscape["Pre"] + epsilon)
    )

    landscape["log2fc_neg_pre"] = np.log2(
        (landscape["Neg"] + epsilon) /
        (landscape["Pre"] + epsilon)
    )

    # Raw frequency shifts relative to the pre-selection population.
    landscape["delta_pos"] = landscape["Pos"] - landscape["Pre"]
    landscape["delta_neg"] = landscape["Neg"] - landscape["Pre"]

    landscape["mut_count"] = landscape["wo"].str.count("O")

    # Pos-vs-Neg enrichment contrast.
    landscape["ddelta"] = (
        landscape["log2fc_pos_pre"] -
        landscape["log2fc_neg_pre"]
    )

    landscape = landscape[
        [
            "library_id",
            "wo",
            "Neg",
            "Pos",
            "Pre",
            "log2fc_pos_pre",
            "log2fc_neg_pre",
            "delta_pos",
            "delta_neg",
            "mut_count",
            "ddelta",
            "count_pre",
            "count_pos",
            "count_neg",
        ]
    ].sort_values(["library_id", "wo"]).reset_index(drop=True)

    return pooled, landscape

In [5]:
baseline_pooled_recalc, baseline_landscape_recalc = recalculate_landscape_from_counts(
    wo_counts,
    epsilon=1e-6
)

baseline_compare = baseline_landscape.merge(
    baseline_landscape_recalc,
    on=["library_id", "wo"],
    how="outer",
    suffixes=("_expected", "_recalculated"),
    indicator=True
)

print(baseline_compare["_merge"].value_counts())

for col in [
    "Neg",
    "Pos",
    "Pre",
    "log2fc_pos_pre",
    "log2fc_neg_pre",
    "delta_pos",
    "delta_neg",
    "ddelta",
]:
    diff = (
        baseline_compare[f"{col}_expected"] -
        baseline_compare[f"{col}_recalculated"]
    ).abs().max()
    print(f"{col}: max absolute difference = {diff}")

_merge
both          768
left_only       0
right_only      0
Name: count, dtype: int64
Neg: max absolute difference = 9.974659986866641e-17
Pos: max absolute difference = 9.985502008591496e-17
Pre: max absolute difference = 9.974659986866641e-17
log2fc_pos_pre: max absolute difference = 4.440892098500626e-16
log2fc_neg_pre: max absolute difference = 4.440892098500626e-16
delta_pos: max absolute difference = 9.985502008591496e-17
delta_neg: max absolute difference = 9.996344030316351e-17
ddelta: max absolute difference = 4.440892098500626e-16


In [6]:
cutoffs = [0, 1_000, 10_000, 100_000]

cutoff_description = pd.DataFrame({
    "cutoff_total_umi_count": cutoffs,
    "interpretation": [
        "Baseline: include all samples present in wo_counts.csv",
        "Exclude very low-depth samples below 1,000 total UMI counts",
        "Exclude low-depth samples below 10,000 total UMI counts",
        "Exclude samples below 100,000 total UMI counts",
    ]
})

cutoff_description

,cutoff_total_umi_count,interpretation
0,0,Baseline: include all samples present in wo_co...
1,1000,"Exclude very low-depth samples below 1,000 tot..."
2,10000,"Exclude low-depth samples below 10,000 total U..."
3,100000,"Exclude samples below 100,000 total UMI counts"


In [7]:
sample_filter_summary = []

for cutoff in cutoffs:
    retained_samples = sample_coverage.loc[
        sample_coverage["total_umi_count"] >= cutoff,
        "sample"
    ]

    excluded = sample_coverage.loc[
        sample_coverage["total_umi_count"] < cutoff
    ].copy()

    retained = sample_coverage.loc[
        sample_coverage["total_umi_count"] >= cutoff
    ].copy()

    sample_filter_summary.append({
        "cutoff_total_umi_count": cutoff,
        "n_samples_retained": retained.shape[0],
        "n_samples_excluded": excluded.shape[0],
        "excluded_samples": "; ".join(excluded["sample"].tolist())
    })

sample_filter_summary = pd.DataFrame(sample_filter_summary)
sample_filter_summary

,cutoff_total_umi_count,n_samples_retained,n_samples_excluded,excluded_samples
0,0,22,0,
1,1000,20,2,Library-2-Pos-2; Library-3-Pre-2
2,10000,19,3,Library-2-Pos-2; Library-3-Pre-2; Library-2-Pre-2
3,100000,16,6,Library-2-Pos-2; Library-3-Pre-2; Library-2-Pr...


In [8]:
pd.set_option("display.max_colwidth", None)
sample_filter_summary

,cutoff_total_umi_count,n_samples_retained,n_samples_excluded,excluded_samples
0,0,22,0,
1,1000,20,2,Library-2-Pos-2; Library-3-Pre-2
2,10000,19,3,Library-2-Pos-2; Library-3-Pre-2; Library-2-Pre-2
3,100000,16,6,Library-2-Pos-2; Library-3-Pre-2; Library-2-Pre-2; Library-1-Neg-2; Library-3-Pos-1; Library-1-Pre-2


In [9]:
filtered_results = {}
summary_rows = []

for cutoff in cutoffs:
    retained_samples = sample_coverage.loc[
        sample_coverage["total_umi_count"] >= cutoff,
        "sample"
    ]

    filtered_counts = wo_counts[wo_counts["sample"].isin(retained_samples)].copy()

    pooled_filtered, landscape_filtered = recalculate_landscape_from_counts(
        filtered_counts,
        epsilon=1e-6
    )

    filtered_results[cutoff] = {
        "counts": filtered_counts,
        "pooled": pooled_filtered,
        "landscape": landscape_filtered,
    }

    summary_rows.append({
        "cutoff_total_umi_count": cutoff,
        "n_samples_retained": filtered_counts["sample"].nunique(),
        "n_rows_counts": filtered_counts.shape[0],
        "n_pooled_rows": pooled_filtered.shape[0],
        "n_landscape_rows": landscape_filtered.shape[0],
        "n_libraries": landscape_filtered["library_id"].nunique(),
        "n_genotypes_total": landscape_filtered[["library_id", "wo"]].drop_duplicates().shape[0],
    })

filter_run_summary = pd.DataFrame(summary_rows)
filter_run_summary

,cutoff_total_umi_count,n_samples_retained,n_rows_counts,n_pooled_rows,n_landscape_rows,n_libraries,n_genotypes_total
0,0,22,5451,2304,768,3,768
1,1000,20,5105,2304,768,3,768
2,10000,19,4857,2304,768,3,768
3,100000,16,4096,2304,768,3,768


In [10]:
comparison_rows = []

compare_cols = [
    "Neg",
    "Pos",
    "Pre",
    "log2fc_pos_pre",
    "log2fc_neg_pre",
    "delta_pos",
    "delta_neg",
    "ddelta",
]

baseline_for_compare = baseline_landscape.sort_values(
    ["library_id", "wo"]
).reset_index(drop=True)

for cutoff, result in filtered_results.items():
    landscape_filtered = result["landscape"]

    merged = baseline_for_compare.merge(
        landscape_filtered,
        on=["library_id", "wo"],
        how="inner",
        suffixes=("_baseline", "_filtered")
    )

    row = {
        "cutoff_total_umi_count": cutoff,
        "n_rows_compared": merged.shape[0],
    }

    for col in compare_cols:
        baseline_col = f"{col}_baseline"
        filtered_col = f"{col}_filtered"

        diff = merged[filtered_col] - merged[baseline_col]

        row[f"{col}_max_abs_diff"] = diff.abs().max()
        row[f"{col}_mean_abs_diff"] = diff.abs().mean()
        row[f"{col}_pearson_r"] = merged[[baseline_col, filtered_col]].corr().iloc[0, 1]

    comparison_rows.append(row)

filter_comparison_summary = pd.DataFrame(comparison_rows)
filter_comparison_summary

,cutoff_total_umi_count,n_rows_compared,Neg_max_abs_diff,Neg_mean_abs_diff,Neg_pearson_r,Pos_max_abs_diff,Pos_mean_abs_diff,Pos_pearson_r,Pre_max_abs_diff,Pre_mean_abs_diff,...,log2fc_neg_pre_pearson_r,delta_pos_max_abs_diff,delta_pos_mean_abs_diff,delta_pos_pearson_r,delta_neg_max_abs_diff,delta_neg_mean_abs_diff,delta_neg_pearson_r,ddelta_max_abs_diff,ddelta_mean_abs_diff,ddelta_pearson_r
0,0,768,9.974660e-17,4.772566e-17,1.0,9.985502e-17,5.008058e-17,1.0,9.974660e-17,4.862096e-17,...,1.0,9.985502e-17,4.626964e-17,1.0,9.996344e-17,4.634737e-17,1.0,4.440892e-16,2.481044e-17,1.0
1,1000,768,9.974660e-17,4.772566e-17,1.0,1.449428e-07,5.929284e-09,1.0,2.661773e-07,1.360591e-08,...,1.0,2.661773e-07,1.953519e-08,1.0,2.661773e-07,1.360591e-08,1.0,7.936754e-04,4.623692e-06,1.0
2,10000,768,9.974660e-17,4.772566e-17,1.0,1.449428e-07,5.929284e-09,1.0,7.113959e-07,6.599518e-08,...,1.0,6.888946e-07,6.639393e-08,1.0,7.113959e-07,6.599518e-08,1.0,7.936754e-04,4.623692e-06,1.0
3,100000,768,3.763506e-06,1.600222e-07,1.0,1.490601e-05,5.349999e-07,1.0,1.013304e-05,3.493334e-07,...,1.0,1.497080e-05,8.664624e-07,1.0,6.369536e-06,3.280377e-07,1.0,8.020144e-03,3.451892e-04,1.0


In [11]:
mutational_load_rows = []

for cutoff, result in filtered_results.items():
    landscape_filtered = result["landscape"]

    grouped = (
        landscape_filtered
        .groupby(["library_id", "mut_count"], as_index=False)
        .agg(
            mean_log2fc_pos_pre=("log2fc_pos_pre", "mean"),
            mean_log2fc_neg_pre=("log2fc_neg_pre", "mean"),
            mean_ddelta=("ddelta", "mean"),
            sd_ddelta=("ddelta", "std"),
            n_genotypes=("wo", "size")
        )
    )

    grouped["cutoff_total_umi_count"] = cutoff
    mutational_load_rows.append(grouped)

mutational_load_summary = pd.concat(mutational_load_rows, ignore_index=True)

mutational_load_summary.head(20)

,library_id,mut_count,mean_log2fc_pos_pre,mean_log2fc_neg_pre,mean_ddelta,sd_ddelta,n_genotypes,cutoff_total_umi_count
0,1,0,0.253572,-0.005690,0.259262,NaN,1,0
1,1,1,-0.391114,-0.092257,-0.298857,0.903527,8,0
2,1,2,-0.759975,-0.218827,-0.541148,1.510677,28,0
3,1,3,-1.312145,-0.333162,-0.978982,1.621195,56,0
4,1,4,-1.096031,-0.281362,-0.814669,1.773891,70,0
5,1,5,-0.477661,-0.028474,-0.449187,1.655273,56,0
6,1,6,-0.026452,0.083640,-0.110092,1.442777,28,0
7,1,7,0.061617,0.730691,-0.669074,1.633468,8,0
8,1,8,1.401357,0.870073,0.531285,NaN,1,0
9,2,0,0.507641,-0.210163,0.717804,NaN,1,0


In [12]:
sample_filter_summary.to_csv(
    CHECKPOINT_DIR / "low_read_sample_filter_summary.csv",
    index=False
)

filter_run_summary.to_csv(
    CHECKPOINT_DIR / "low_read_filter_run_summary.csv",
    index=False
)

filter_comparison_summary.to_csv(
    CHECKPOINT_DIR / "low_read_filter_landscape_comparison_summary.csv",
    index=False
)

mutational_load_summary.to_csv(
    CHECKPOINT_DIR / "low_read_filter_mutational_load_summary.csv",
    index=False
)

for cutoff, result in filtered_results.items():
    result["pooled"].to_csv(
        CHECKPOINT_DIR / f"pooled_counts_min_sample_umi_{cutoff}.csv",
        index=False
    )

    result["landscape"].to_csv(
        CHECKPOINT_DIR / f"landscape_min_sample_umi_{cutoff}.csv",
        index=False
    )

print("Saved low-read filter sensitivity outputs to:", CHECKPOINT_DIR)

Saved low-read filter sensitivity outputs to: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/outputs/checkpoints


## Sensitivity analysis summary

This notebook recalculates the W/O genotype landscape after excluding sequencing samples below several sample-level total UMI count thresholds.

The baseline cutoff of `0` corresponds to the dissertation-associated processed analysis using all samples present in `wo_counts.csv`.

The tested sample-level cutoffs were:

- `0` total UMI counts: baseline; all samples present in `wo_counts.csv` retained
- `1,000` total UMI counts: excludes the two lowest-depth included samples
- `10,000` total UMI counts: excludes the three lowest-depth included samples
- `100,000` total UMI counts: excludes six lower-depth samples

Across all tested cutoffs, the filtered recalculations retained the full 3-library by 256-genotype W/O landscape structure. The 1,000 and 10,000 total UMI count cutoffs produced only negligible differences relative to the dissertation-associated baseline. Even the more aggressive 100,000 total UMI count cutoff preserved the overall landscape structure and showed essentially perfect correlation with the baseline values, although absolute differences were larger.

These results indicate that the main W/O landscape calculations are robust to removal of low-depth samples. Final cutoff selection should be determined based on the agreed QC threshold for sample inclusion.